# F1 Moodle Activity: Predicting Students at Risk

## Modeling

This notebook develops models for identifying students who are at risk of failing the course based on Moodle activity and early homework performance.

The features are prepared using the project data preparation pipeline. All features use information available before the cutoff, which is defined as one week before the first midterm.

The models are trained on students from 2019 and 2021–2024 and tested on students from 2025–2026.

In [1]:
import sys
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

sys.path.append("../src")
from prepare import build_features

## 1. Data preparation

The final feature table is created by `build_features()` from the project data.

One row represents one student in one academic year.

The target variable is `fail`:

- 1 = F grade or no grade
- 0 = A–E grade

The table contains activity features and early homework features calculated using only information available before the cutoff.

In [2]:
import sys

sys.path.append("../src")

from prepare import build_features

data = build_features("../data")

print(data.shape)
print(data.columns.tolist())
display(data.head())

(843, 17)
['year', 'actor_id', 'fail', 'is_repeat', 'active_days', 'active_weeks', 'first_day_lag', 'active_weeks_share', 'active_days_last_2_weeks', 'days_forum', 'days_grades', 'days_homework', 'days_lectures', 'days_materials', 'hw_early_count', 'hw_early_submitted_share', 'hw_early_mean']


,year,actor_id,fail,is_repeat,active_days,active_weeks,first_day_lag,active_weeks_share,active_days_last_2_weeks,days_forum,days_grades,days_homework,days_lectures,days_materials,hw_early_count,hw_early_submitted_share,hw_early_mean
0,2019,S027cf9,0,0,18,6,19,0.666667,5,0,1,18,0,4,5,1.0,83.8
1,2019,S032255,0,0,6,5,21,0.555556,2,1,0,6,0,2,5,0.4,4.4
2,2019,S03a8b8,0,0,30,8,0,0.888889,9,1,16,22,0,10,5,1.0,106.4
3,2019,S04cf7f,0,0,21,9,6,1.000000,7,0,5,18,0,3,5,1.0,94.4
4,2019,S05f285,1,0,32,7,7,0.777778,13,2,25,25,0,12,5,1.0,86.2


### Train and test split

To evaluate whether the model generalizes to newer academic years, the data is split by year instead of randomly.

- Training data: 2019 and 2021–2024
- Test data: 2025–2026
- 2020 is excluded because the prepared dataset does not contain a valid target for that year.

In [3]:
train = data[
    data["year"].isin([2019, 2021, 2022, 2023, 2024])
].copy()

test = data[
    data["year"].isin([2025, 2026])
].copy()

print("Train:", train.shape)
print("Test:", test.shape)

print("\nTrain years:")
print(train["year"].value_counts().sort_index())

print("\nTest years:")
print(test["year"].value_counts().sort_index())

Train: (564, 17)
Test: (279, 17)

Train years:
year
2019    114
2021    142
2022     83
2023    106
2024    119
Name: count, dtype: int64

Test years:
year
2025    137
2026    142
Name: count, dtype: int64


In [4]:
print("Train target distribution:")
print(train["fail"].value_counts())

print("\nTest target distribution:")
print(test["fail"].value_counts())

Train target distribution:
fail
0    482
1     82
Name: count, dtype: int64

Test target distribution:
fail
0    234
1     45
Name: count, dtype: int64


## 2. Baseline

As a simple baseline, we use only `active_days`.

A student is classified as being at risk if they were active on 15 or fewer days before the cutoff.

This is a simple reference rule and does not use the other available features.

In [5]:
baseline_threshold = 15

y_train = train["fail"].astype(int)
y_test = test["fail"].astype(int)

y_train_baseline = (
    train["active_days"] <= baseline_threshold
).astype(int)

y_test_baseline = (
    test["active_days"] <= baseline_threshold
).astype(int)

baseline_train_recall = recall_score(
    y_train,
    y_train_baseline,
    zero_division=0
)

baseline_train_precision = precision_score(
    y_train,
    y_train_baseline,
    zero_division=0
)

baseline_test_recall = recall_score(
    y_test,
    y_test_baseline,
    zero_division=0
)

baseline_test_precision = precision_score(
    y_test,
    y_test_baseline,
    zero_division=0
)

print("Baseline threshold:", baseline_threshold)

print("\nTrain recall:", baseline_train_recall)
print("Train precision:", baseline_train_precision)

print("\nTest recall:", baseline_test_recall)
print("Test precision:", baseline_test_precision)

Baseline threshold: 15

Train recall: 0.6219512195121951
Train precision: 0.3617021276595745

Test recall: 0.7333333333333333
Test precision: 0.358695652173913


### Baseline result

The baseline provides a simple reference for the machine learning models.

Its performance on the 2025–2026 test data will be compared with Logistic Regression and Random Forest.

In [6]:
features = [
    "is_repeat",
    "active_days",
    "active_weeks",
    "active_weeks_share",
    "first_day_lag",
    "active_days_last_2_weeks",
    "days_materials",
    "days_homework",
    "days_lectures",
    "days_forum",
    "days_grades",
    "hw_early_count",
    "hw_early_submitted_share",
    "hw_early_mean"
]

X_train = train[features].copy()
X_test = test[features].copy()

y_train = train["fail"].astype(int)
y_test = test["fail"].astype(int)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("Number of features:", len(features))
print("Features:")
print(features)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

Number of features: 14
Features:
['is_repeat', 'active_days', 'active_weeks', 'active_weeks_share', 'first_day_lag', 'active_days_last_2_weeks', 'days_materials', 'days_homework', 'days_lectures', 'days_forum', 'days_grades', 'hw_early_count', 'hw_early_submitted_share', 'hw_early_mean']

Train shape: (564, 14)
Test shape: (279, 14)


## 4. Logistic Regression

We train a Logistic Regression model using 14 features describing Moodle activity, early homework performance, and the student's previous participation in the course.

The model predicts the probability that a student will fail the course.

All features are calculated using only information available before the prediction cutoff.

In [7]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(
    X_train,
    y_train
)

train_prob_logistic = logistic_model.predict_proba(
    X_train
)[:, 1]

test_prob_logistic = logistic_model.predict_proba(
    X_test
)[:, 1]

print("First 10 Logistic Regression probabilities:")
print(test_prob_logistic[:10])

First 10 Logistic Regression probabilities:
[0.02456458 0.0226633  0.00700081 0.11882149 0.75027398 0.02638228
 0.0327806  0.57501129 0.03831368 0.15662238]


In [8]:
logistic_train_auc = roc_auc_score(
    y_train,
    train_prob_logistic
)

logistic_test_auc = roc_auc_score(
    y_test,
    test_prob_logistic
)

print("Logistic Regression Train AUC:", logistic_train_auc)
print("Logistic Regression Test AUC:", logistic_test_auc)

Logistic Regression Train AUC: 0.9000860236818135
Logistic Regression Test AUC: 0.8286799620132954


### Logistic Regression AUC

The Logistic Regression model achieved a test AUC of approximately 0.828.

The training AUC was approximately 0.901, while the test AUC was approximately 0.828. The difference indicates some overfitting, but the model still provides useful discrimination on the unseen 2025–2026 data.

### Selecting the classification threshold

Logistic Regression produces a probability of failure for each student.

To turn these probabilities into risk labels, we need to choose a classification threshold.

The threshold is selected using only the training data.

The project criteria are:

- recall ≥ 0.70
- precision ≥ 1/3

The test data is not used when choosing the threshold.

In [9]:
logistic_threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):

    y_pred = (
        train_prob_logistic >= threshold
    ).astype(int)

    recall = recall_score(
        y_train,
        y_pred,
        zero_division=0
    )

    precision = precision_score(
        y_train,
        y_pred,
        zero_division=0
    )

    logistic_threshold_results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision
    })

logistic_thresholds = pd.DataFrame(
    logistic_threshold_results
)

display(
    logistic_thresholds[
        (logistic_thresholds["recall"] >= 0.70) &
        (logistic_thresholds["precision"] >= 1/3)
    ]
)

,threshold,recall,precision
3,0.08,0.890244,0.334862
4,0.09,0.865854,0.360406
5,0.10,0.841463,0.372973
6,0.11,0.841463,0.403509
7,0.12,0.817073,0.408537
8,0.13,0.780488,0.412903
9,0.14,0.768293,0.428571
10,0.15,0.768293,0.453237
11,0.16,0.756098,0.469697
12,0.17,0.743902,0.495935


### Threshold selection result

Several thresholds satisfy both project criteria on the training data.

We select a threshold of 0.21. At this threshold, the training recall is approximately 0.707 and the training precision is approximately 0.527.

The threshold is selected using only the training data. The 2025–2026 test data is used only for the final evaluation.

In [10]:
logistic_threshold = 0.21

y_test_pred_logistic = (
    test_prob_logistic >= logistic_threshold
).astype(int)

logistic_test_recall = recall_score(
    y_test,
    y_test_pred_logistic,
    zero_division=0
)

logistic_test_precision = precision_score(
    y_test,
    y_test_pred_logistic,
    zero_division=0
)

print(
    "Logistic Regression threshold:",
    logistic_threshold
)

print(
    "Test recall:",
    logistic_test_recall
)

print(
    "Test precision:",
    logistic_test_precision
)

Logistic Regression threshold: 0.21
Test recall: 0.4222222222222222
Test precision: 0.59375


### Logistic Regression result

The Logistic Regression model achieved a test AUC of approximately 0.828.

The classification threshold was selected using the training data. The selected threshold was 0.21, where the training recall was approximately 0.707 and the training precision was approximately 0.527.

On the 2025–2026 test data, the model achieved a recall of approximately 0.422 and a precision of approximately 0.594.

Therefore, the precision requirement is satisfied, but the target recall of 0.70 is not reached on the test data. This indicates that the model's performance does not transfer perfectly from the historical training years to the 2025–2026 cohorts.

## 5. Random Forest

We train a Random Forest using the same 14 features as the Logistic Regression model.

This allows us to compare the two machine learning approaches using the same information.

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=1
)

rf_model.fit(
    X_train,
    y_train
)

train_prob_rf = rf_model.predict_proba(
    X_train
)[:, 1]

test_prob_rf = rf_model.predict_proba(
    X_test
)[:, 1]

print("First 10 Random Forest probabilities:")
print(test_prob_rf[:10])

First 10 Random Forest probabilities:
[0.   0.   0.02 0.08 0.87 0.25 0.   0.64 0.13 0.21]


In [12]:
rf_train_auc = roc_auc_score(
    y_train,
    train_prob_rf
)

rf_test_auc = roc_auc_score(
    y_test,
    test_prob_rf
)

print("Random Forest Train AUC:", rf_train_auc)
print("Random Forest Test AUC:", rf_test_auc)

Random Forest Train AUC: 1.0
Random Forest Test AUC: 0.795868945868946


### Random Forest feature importance

Feature importance shows which predictors contribute most to the Random Forest model.

In [13]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
13,hw_early_mean,0.184178
12,hw_early_submitted_share,0.119409
7,days_homework,0.109986
5,active_days_last_2_weeks,0.086243
1,active_days,0.085945
3,active_weeks_share,0.068190
4,first_day_lag,0.061985
10,days_grades,0.058196
6,days_materials,0.053471
2,active_weeks,0.050895


### Selecting the Random Forest threshold

The threshold is selected using only the training data.

We look for thresholds satisfying:

- recall ≥ 0.70
- precision ≥ 1/3

The selected threshold is then evaluated once on the 2025–2026 test data.

In [14]:
rf_threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):

    y_pred = (
        train_prob_rf >= threshold
    ).astype(int)

    recall = recall_score(
        y_train,
        y_pred,
        zero_division=0
    )

    precision = precision_score(
        y_train,
        y_pred,
        zero_division=0
    )

    rf_threshold_results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision
    })

rf_thresholds = pd.DataFrame(
    rf_threshold_results
)

display(
    rf_thresholds[
        (rf_thresholds["recall"] >= 0.70) &
        (rf_thresholds["precision"] >= 1/3)
    ]
)

,threshold,recall,precision
0,0.05,1.000000,0.398058
1,0.06,1.000000,0.471264
2,0.07,1.000000,0.471264
3,0.08,1.000000,0.569444
4,0.09,1.000000,0.611940
...,...,...,...
60,0.65,0.841463,1.000000
61,0.66,0.829268,1.000000
62,0.67,0.804878,1.000000
63,0.68,0.804878,1.000000


In [15]:
valid_rf_thresholds = rf_thresholds[
    rf_thresholds["precision"] >= 1/3
]

best_rf_threshold_row = valid_rf_thresholds.sort_values(
    ["recall", "precision"],
    ascending=[False, False]
).iloc[0]

rf_threshold = best_rf_threshold_row["threshold"]

print("Selected Random Forest threshold:", rf_threshold)
print("Train recall:", best_rf_threshold_row["recall"])
print("Train precision:", best_rf_threshold_row["precision"])

Selected Random Forest threshold: 0.36000000000000004
Train recall: 1.0
Train precision: 1.0


In [18]:
y_test_pred_rf = (
    test_prob_rf >= rf_threshold
).astype(int)

rf_test_recall = recall_score(
    y_test,
    y_test_pred_rf,
    zero_division=0
)

rf_test_precision = precision_score(
    y_test,
    y_test_pred_rf,
    zero_division=0
)

print("Random Forest threshold:", rf_threshold)
print("Test recall:", rf_test_recall)
print("Test precision:", rf_test_precision)

Random Forest threshold: 0.36000000000000004
Test recall: 0.37777777777777777
Test precision: 0.5483870967741935


### Random Forest result

The Random Forest achieved a test AUC of approximately 0.796.

The classification threshold was selected using only the training data. The selected threshold was 0.36, where the training recall and precision were both 1.000.

On the 2025–2026 test data, the model achieved a recall of approximately 0.378 and a precision of approximately 0.548.

Therefore, the precision requirement is satisfied, but the target recall of 0.70 is not reached on the test data. The large difference between training and test performance indicates strong overfitting and limited generalization to the newer cohorts.

## 6. Final model comparison

The models are evaluated on the 2025–2026 test data.

The main project goal is to identify at least 70% of students who fail while maintaining a precision of at least 1/3.

In [20]:
comparison = pd.DataFrame({
    "model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "test_auc": [
        np.nan,
        logistic_test_auc,
        rf_test_auc
    ],
    "test_recall": [
        0.7333333333,
        logistic_test_recall,
        rf_test_recall
    ],
    "test_precision": [
        0.3586956522,
        logistic_test_precision,
        rf_test_precision
    ]
})

display(comparison)

,model,test_auc,test_recall,test_precision
0,Baseline,NaN,0.733333,0.358696
1,Logistic Regression,0.828680,0.422222,0.593750
2,Random Forest,0.795869,0.377778,0.548387


## Interpretation of feature importance

The Random Forest feature importance analysis shows that the most important predictors are `hw_early_mean`, `hw_early_submitted_share`, `days_homework`, `active_days_last_2_weeks`, and `active_days`.

This suggests that early homework performance and recent Moodle activity contain useful information for identifying students at risk of failing.

However, feature importance should be interpreted carefully because the meaning of Moodle activity can differ between academic years.

## Final results and discussion

The baseline rule based on Moodle activity achieved the highest recall on the 2025–2026 test data: 0.733. Its precision was 0.359, which is slightly above the required minimum of 1/3.

Logistic Regression achieved the highest test AUC (0.829) among the evaluated models. Its test precision was 0.594, but its recall was only 0.422. Random Forest achieved a test AUC of 0.796, with a test recall of 0.378 and precision of 0.548.

Therefore, neither machine-learning model reached the target recall of 0.70 on the test data. The simple baseline rule performed better for the main goal of identifying students who eventually fail.

The difference between training and test performance suggests that the models do not generalize perfectly from the historical cohorts to the 2025–2026 cohorts. Differences between years may affect the meaning of Moodle activity features. In particular, the availability and use of Moodle tools changed over time, and students may also become more active shortly before the midterm, after the prediction cutoff.

Feature importance from the Random Forest showed that `hw_early_mean`, `hw_early_submitted_share`, and `days_homework` were among the most important features. This suggests that early homework performance and homework-related activity contain useful information for predicting failure.

Overall, the results show that Moodle activity can provide useful early warning information, but model performance depends strongly on the cohort and the period in which the prediction is made.

## Limitations

There are several limitations to the prediction task.

First, Moodle usage patterns changed between academic years. Some tools were not available or were used differently in different years, so the same feature may not have exactly the same meaning across cohorts.

Second, the prediction cutoff is one week before the first midterm. Students may become much more active after this cutoff while preparing for the midterm. Such activity cannot be used by the model because the goal is to provide an early warning.

Third, the test data consists of newer cohorts from 2025–2026, while the models are trained on earlier cohorts. The lower test recall therefore suggests that changes between cohorts are important and should be considered when interpreting the results.